In [1]:
import pandas as pd
import json


In [2]:
import json
import pandas as pd

data = []
with open("../data/problems.jsonl", "r") as f:
    for line in f:
        # Remove outer quotes and parse inner JSON
        inner_json = line.strip()[1:-1]  # Assumes lines are quoted strings
        inner_json = inner_json.replace('""', '"')  # Fix escaping
        data.append(json.loads(inner_json))

df = pd.DataFrame(data)

In [3]:
df.columns


Index(['title', 'description', 'input_description', 'output_description',
       'sample_io', 'problem_class', 'problem_score', 'url'],
      dtype='object')

In [4]:
# Handle missing values
df.fillna("", inplace=True)

# Combine text fields
df["full_text"] = (
    df["title"] + " " +
    df["description"] + " " +
    df["input_description"] + " " +
    df["output_description"]
)


In [5]:
df["full_text"].iloc[0][:400]


'Uuu Unununium (Uuu) was the name of the chemical\n    element with atom number 111, until it changed to\n    Röntgenium (Rg) in 2004. These heavy elements are very\n    unstable and have only been synthesized in a few\n    laboratories.\nYou have just been hired by one of these labs to optimize\n    the algorithms used in simulations. For example, when\n    simulating complicated chemical reactions, it i'

In [6]:
X_text = df["full_text"]

y_class = df["problem_class"]
y_score = df["problem_score"]


In [7]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_class_train, y_class_test = train_test_split(
    X_text, y_class, test_size=0.2, random_state=42, stratify=y_class
)

_, _, y_score_train, y_score_test = train_test_split(
    X_text, y_score, test_size=0.2, random_state=42
)


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    stop_words="english"
)

X_train = vectorizer.fit_transform(X_train_text)
X_test  = vectorizer.transform(X_test_text)


In [9]:
from sklearn.svm import LinearSVC

clf = LinearSVC()
clf.fit(X_train, y_class_train)


y_pred_class = clf.predict(X_test)


In [10]:
from sklearn.metrics import accuracy_score, confusion_matrix

print("Accuracy:", accuracy_score(y_class_test, y_pred_class))
print(confusion_matrix(y_class_test, y_pred_class))


Accuracy: 0.479951397326853
[[ 47  59  47]
 [ 39 255  95]
 [ 28 160  93]]


In [11]:
from sklearn.ensemble import RandomForestRegressor

reg = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

reg.fit(X_train, y_score_train)
y_pred_score = reg.predict(X_test)


In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

print("MAE:", mean_absolute_error(y_score_test, y_pred_score))
print("RMSE:", np.sqrt(mean_squared_error(y_score_test, y_pred_score)))

MAE: 1.8944292223572294
RMSE: 2.230126312384673


In [ ]:
import pickle
import os

# ensure models folder exists
os.makedirs("../models", exist_ok=True)

# save models
with open("../models/classifier.pkl", "wb") as f:
    pickle.dump(clf, f)

with open("../models/regressor.pkl", "wb") as f:
    pickle.dump(reg, f)

with open("../models/vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("✅ Models saved successfully")


✅ Models saved successfully
